# Innerbloom Phase 3: a light model for reading entries

This trains a **small** model (Qwen2.5 1.5B by default) to do one job: read a journal entry and return Innerbloom's JSON (emotion, intensity, summary, tags, themes, people). The goal is to match the 7B model that Innerbloom uses today, while running fast on an ordinary laptop.

**Before you start**
- Runtime > Change runtime type > **A100 GPU**. Check the compute units left in the resource panel (top right).
- Run the cells **one at a time, top to bottom**. Each says what it costs. If anything fails, stop and send the error.
- Everything is saved to your Google Drive, so if Colab disconnects, run the same cell again and it continues.
- Only fictional training data is used. No journal of yours leaves your Mac.
- **When you finish, Runtime > Disconnect and delete runtime**, or it keeps using units.

## 1. Check the GPU (free, seconds)
You should see an **A100** in the table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive (free)
A window asks for permission. Results go to `MyDrive/innerbloom/phase3`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/innerbloom/phase3'
import os; os.makedirs(OUT, exist_ok=True); print('saving to', OUT)

Mounted at /content/drive
saving to /content/drive/MyDrive/innerbloom/phase3


## 3. Get the code and libraries (about 2 minutes)
Downloads Innerbloom from GitHub and installs the pinned training libraries.

In [2]:
BRANCH = 'feature/innerbloom-ai-v2'
!rm -rf /content/innerbloom && git clone -q --depth 1 -b $BRANCH https://github.com/GhanshyamPaunikar/innerbloom /content/innerbloom
%cd /content/innerbloom
!git log --oneline -1
!pip install -q -r ml/requirements-train.txt

/content/innerbloom
55461f0 (grafted, HEAD -> feature/innerbloom-ai-v2, origin/feature/innerbloom-ai-v2) Round 2: 160 training-only contrast pairs and tagged retraining options
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 4. Check everything before spending units (about 1 minute)
Confirms the GPU, the library versions, that the training data passes its checks, and that the longest entry fits. It ends with a line of numbers. If it stops with an error, send it.

In [ ]:
!python -m ml.train_emotion --stage check --out $OUT

## 5. Prepare the training answers (one time, roughly 40-70 minutes, about 6-10 units)
My labels cover emotion, intensity and names. The summary, tags and themes come from the 7B model, so the small model learns the whole format. This runs once and is shared by every size. If it disconnects, run it again: it continues.

In [ ]:
!python -m ml.train_emotion --stage replies --out $OUT

## 6. Smoke test (about 5 minutes, under 1 unit)
Ten training steps, then saves to Drive, reloads, and reads three entries. You should see **SMOKE TEST PASSED**. Do not go on until you do.

In [ ]:
BASE = '1.5b'   # light model; '3b' and '0.5b' are for comparison
!python -m ml.train_emotion --stage smoke --base $BASE --out $OUT

## 7. Train (roughly 15-40 minutes for 1.5B, about 2-6 units)
Up to 3 passes over 1,199 entries, checking the 402 validation entries every 20 steps and stopping early if it stops improving. Saved to Drive as it goes; if it disconnects, run the cell again to resume.

In [ ]:
!python -m ml.train_emotion --stage train --base $BASE --out $OUT

## 8. Compare (roughly 15-30 minutes, about 2-4 units)
Reads the 402 validation entries with the untrained and the trained small model, and compares both with today's 7B. The last lines say whether to keep it: **keep** (better than untrained, nothing worse) and **can_replace_7b_for_reading**.

In [ ]:
!python -m ml.train_emotion --stage evaluate --base $BASE --out $OUT

## 8b. Export for your Mac (roughly 5-10 minutes, about 1 unit)
Merges the trained add-on into the model and writes **one file** Ollama can run: `innerbloom-reader-1.5b-f16.gguf` (about 3 GB) in `MyDrive/innerbloom/phase3/qwen2.5-1.5b/`. Download it to your Mac's Downloads folder.

In [3]:
!python -m ml.train_emotion --stage export --base $BASE --out $OUT

usage: train_emotion.py [-h]
                        --stage {check,replies,smoke,train,evaluate,export}
                        [--out OUT] [--base {0.5b,1.5b,3b,7b}] [--tag TAG]
                        [--lora-r LORA_R] [--lr LR]
train_emotion.py: error: argument --base: expected one argument


## Round 2: sharper distinctions (about 3-5 units in all)
160 extra training-only entries, written as contrast pairs (proud vs happy, tired vs overwhelmed, angry or frustrated vs sad, content or grateful vs happy, sarcasm). Validation and test are unchanged. Round 1 stays in its own folder.

**R2-a. New answers for the 160 entries only** (a few minutes: it skips the ones already done).

In [4]:
!python -m ml.train_emotion --stage replies --out $OUT

tokenizer_config.json: 7.30kB [00:00, 22.6MB/s]
vocab.json: 2.78MB [00:00, 60.6MB/s]
merges.txt: 1.67MB [00:00, 139MB/s]
tokenizer.json: 7.03MB [00:00, 193MB/s]
config.json: 100% 663/663 [00:00<00:00, 6.80MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1794127488.729279    3427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794127488.794143    3427 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794127491.334998    3427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to flo

**R2-b. Train round 2** (roughly 10-20 minutes): larger add-on (rank 32), gentler learning rate.

In [5]:
BASE = '1.5b'
!python -m ml.train_emotion --stage train --base $BASE --tag r2 --lora-r 32 --lr 1e-4 --out $OUT

tokenizer_config.json: 7.30kB [00:00, 27.3MB/s]
vocab.json: 2.78MB [00:00, 61.5MB/s]
merges.txt: 1.67MB [00:00, 130MB/s]
tokenizer.json: 7.03MB [00:00, 182MB/s]
config.json: 100% 660/660 [00:00<00:00, 3.18MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1794127665.882866    4401 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794127665.949992    4401 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794127668.484509    4401 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to flo

**R2-c. Compare** (roughly 10 minutes). Then **R2-d. Export** for the Mac.

In [6]:
!python -m ml.train_emotion --stage evaluate --base $BASE --tag r2 --out $OUT

`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1794128375.739921    7655 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794128375.810948    7655 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794128378.579335    7655 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794128378.580108    7655 cudart_stub.cc:31] Could not find c

In [7]:
!python -m ml.train_emotion --stage export --base $BASE --tag r2 --out $OUT

I0000 00:00:1794128980.784357   10348 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794128980.856753   10348 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794128983.305997   10348 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794128983.306535   10348 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

## 9. Finish
1. Copy everything printed by step 8 and send it back, with the units used (resource panel).
2. **Runtime > Disconnect and delete runtime.**

The trained add-on (`adapter`) stays in Drive. Turning it into a model Ollama can run on your Mac is Phase 6. The locked test set is not used here.